In [1]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Input, Conv1D, Flatten, Dense, Permute
from tensorflow.keras.regularizers import l2

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from numpy.lib.stride_tricks import as_strided

I0000 00:00:1788832533.229244   40896 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Loading CSV

In [2]:
DATASET_DIR = "../dataset/phase1"
CMS_TAG = "cms3"
SEQ_LEN = 275
DIVISOR = 200000.0
NUM_CLASSES = 2
CLASSES = np.array(['benign', 'malware'])

def load_data():
    bdir = os.path.join(DATASET_DIR, f'benign_{CMS_TAG}')
    mdir = os.path.join(DATASET_DIR, f'malware_{CMS_TAG}')
    bfiles = sorted(glob.glob(os.path.join(bdir, 'cms_*.csv')))
    mfiles = sorted(glob.glob(os.path.join(mdir, 'cms_*.csv')))

    def load_all(files):
        out = []
        for f in files:
            v = pd.read_csv(f, header=None).values.flatten().astype(np.float32)
            out.append(v)
        return np.array(out, dtype=np.float32)

    Xb = load_all(bfiles)
    Xm = load_all(mfiles)
    X = np.concatenate([Xb, Xm], axis=0)
    y = np.concatenate([np.zeros(len(Xb), dtype=np.int64), np.ones(len(Xm), dtype=np.int64)])
    return X, y

In [3]:
X, y = load_data()

## Train, Validation, Test Split and Normalize

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

X_train = np.clip(X_train / DIVISOR, 0.0, 1.0)
X_val = np.clip(X_val / DIVISOR, 0.0, 1.0)
X_test = np.clip(X_test / DIVISOR, 0.0, 1.0)

X_train = X_train.reshape(-1, SEQ_LEN, 1)
X_val = X_val.reshape(-1, SEQ_LEN, 1)
X_test = X_test.reshape(-1, SEQ_LEN, 1)

## 1D CNN model

In [5]:
input_layer = Input(shape=(SEQ_LEN, 1))

x = Conv1D(filters=16, kernel_size=3, strides=10, padding='valid', activation='relu')(input_layer)
x = Conv1D(filters=32, kernel_size=3, strides=1, padding='valid', activation='relu')(x)
x = Conv1D(filters=64, kernel_size=3, strides=1, padding='valid', activation='relu')(x)

x = Permute((2, 1))(x)
x = Flatten()(x)

x = Dense(32, activation='relu', kernel_regularizer=l2(1e-4))(x)
output_layer = Dense(NUM_CLASSES, activation='softmax', kernel_regularizer=l2(1e-4))(x)

model = Model(input_layer, output_layer)

opt = Adam(learning_rate=0.001)
model.compile(loss='sparse_categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

## Check Point

In [6]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=0.00001)
checkpoint = ModelCheckpoint(
    filepath='./phase1_cms3.h5',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

## Model Training

In [7]:
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val), callbacks=[reduce_lr, checkpoint])

Epoch 1/100
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8456 - loss: 0.4820
Epoch 1: val_accuracy improved from None to 0.87034, saving model to ./phase1_cms3.h5



Epoch 1: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8458 - loss: 0.4776 - val_accuracy: 0.8703 - val_loss: 0.3111 - learning_rate: 0.0010
Epoch 2/100
67/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8787 - loss: 0.3047
Epoch 2: val_accuracy did not improve from 0.87034
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8769 - loss: 0.3050 - val_accuracy: 0.8632 - val_loss: 0.2735 - learning_rate: 0.0010
Epoch 3/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8866 - loss: 0.2530
Epoch 3: val_accuracy improved from 0.87034 to 0.89343, saving model to ./phase1_cms3.h5



Epoch 3: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8867 - loss: 0.2527 - val_accuracy: 0.8934 - val_loss: 0.2062 - learning_rate: 0.0010
Epoch 4/100
64/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9194 - loss: 0.2016
Epoch 4: val_accuracy improved from 0.89343 to 0.93250, saving model to ./phase1_cms3.h5



Epoch 4: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9191 - loss: 0.2025 - val_accuracy: 0.9325 - val_loss: 0.1762 - learning_rate: 0.0010
Epoch 5/100
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9279 - loss: 0.1901
Epoch 5: val_accuracy improved from 0.93250 to 0.94494, saving model to ./phase1_cms3.h5



Epoch 5: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9294 - loss: 0.1887 - val_accuracy: 0.9449 - val_loss: 0.1679 - learning_rate: 0.0010
Epoch 6/100
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9341 - loss: 0.1662
Epoch 6: val_accuracy did not improve from 0.94494
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9329 - loss: 0.1706 - val_accuracy: 0.9432 - val_loss: 0.1634 - learning_rate: 0.0010
Epoch 7/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9353 - loss: 0.1733
Epoch 7: val_accuracy improved from 0.94494 to 0.94671, saving model to ./phase1_cms3.h5



Epoch 7: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9356 - loss: 0.1729 - val_accuracy: 0.9467 - val_loss: 0.1501 - learning_rate: 0.0010
Epoch 8/100
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9403 - loss: 0.1625
Epoch 8: val_accuracy did not improve from 0.94671
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9400 - loss: 0.1633 - val_accuracy: 0.9343 - val_loss: 0.1536 - learning_rate: 0.0010
Epoch 9/100
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9399 - loss: 0.1540
Epoch 9: val_accuracy improved from 0.94671 to 0.95204, saving model to ./phase1_cms3.h5



Epoch 9: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9387 - loss: 0.1552 - val_accuracy: 0.9520 - val_loss: 0.1513 - learning_rate: 0.0010
Epoch 10/100
60/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9406 - loss: 0.1537
Epoch 10: val_accuracy did not improve from 0.95204
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9422 - loss: 0.1527 - val_accuracy: 0.9503 - val_loss: 0.1488 - learning_rate: 0.0010
Epoch 11/100
67/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9380 - loss: 0.1578
Epoch 11: val_accuracy did not improve from 0.95204
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9396 - loss: 0.1543 - val_accuracy: 0.9449 - val_loss: 0.1421 - learning_rate: 0.0010
Epoch 12/100
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9466 - loss: 0.1458
Epoch 12: val_accuracy improved from 0.95204 to 0.95737, saving model to ./phase1_cms3.h5



Epoch 12: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9462 - loss: 0.1457 - val_accuracy: 0.9574 - val_loss: 0.1370 - learning_rate: 0.0010
Epoch 13/100
67/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9440 - loss: 0.1512
Epoch 13: val_accuracy did not improve from 0.95737
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9431 - loss: 0.1518 - val_accuracy: 0.9538 - val_loss: 0.1406 - learning_rate: 0.0010
Epoch 14/100
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9425 - loss: 0.1476
Epoch 14: val_accuracy did not improve from 0.95737
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9418 - loss: 0.1508 - val_accuracy: 0.9556 - val_loss: 0.1427 - learning_rate: 0.0010
Epoch 15/100
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9485 - loss: 0.1429
Epoch 15: val_accuracy did not improve from 0.95737
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9480 - loss: 0.1440 - val_accuracy: 0.9556 - val_loss: 0.1353 -


Epoch 29: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9560 - loss: 0.1250 - val_accuracy: 0.9591 - val_loss: 0.1250 - learning_rate: 5.0000e-04
Epoch 30/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9522 - loss: 0.1257
Epoch 30: val_accuracy did not improve from 0.95915
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9520 - loss: 0.1269 - val_accuracy: 0.9556 - val_loss: 0.1260 - learning_rate: 5.0000e-04
Epoch 31/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9547 - loss: 0.1228
Epoch 31: val_accuracy did not improve from 0.95915
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9547 - loss: 0.1228 - val_accuracy: 0.9556 - val_loss: 0.1246 - learning_rate: 5.0000e-04
Epoch 32/100
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9573 - loss: 0.1220
Epoch 32: val_accuracy did not improve from 0.95915
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9574 - loss: 0.1235 - val_accuracy: 0.9574 - val_lo


Epoch 41: finished saving model to ./phase1_cms3.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9560 - loss: 0.1205 - val_accuracy: 0.9645 - val_loss: 0.1246 - learning_rate: 5.0000e-04
Epoch 42/100
66/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9588 - loss: 0.1188
Epoch 42: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9582 - loss: 0.1225 - val_accuracy: 0.9591 - val_loss: 0.1184 - learning_rate: 5.0000e-04
Epoch 43/100
67/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9566 - loss: 0.1229
Epoch 43: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9578 - loss: 0.1208 - val_accuracy: 0.9574 - val_loss: 0.1186 - learning_rate: 5.0000e-04
Epoch 44/100
67/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9590 - loss: 0.1200
Epoch 44: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9582 - loss: 0.1201 - val_accuracy: 0.9574 - val_lo

## Evaluate (float32)

In [8]:
cp_model = load_model('./phase1_cms3.h5')
cp_model.evaluate(X_test, y_test, batch_size=1000)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9619 - loss: 0.1195 


[0.11945879459381104, 0.9618573784828186]

In [9]:
y_pred = cp_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes, target_names=list(CLASSES), digits=4))

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
              precision    recall  f1-score   support

      benign     0.9405    0.9888    0.9640       623
     malware     0.9873    0.9331    0.9594       583

    accuracy                         0.9619      1206
   macro avg     0.9639    0.9609    0.9617      1206
weighted avg     0.9631    0.9619    0.9618      1206



In [10]:
conf_matrix = confusion_matrix(y_test, y_pred_classes)
conf_matrix_df = pd.DataFrame(conf_matrix, index=list(CLASSES), columns=list(CLASSES))
print("Confusion Matrix:")
print(conf_matrix_df)

Confusion Matrix:
         benign  malware
benign      616        7
malware      39      544


## Q15 Quantization

In [11]:
def scale_for_q15(w):
    absmax = np.abs(w).max()
    if absmax <= 1.0:
        return w, 1
    scale = 1
    while absmax / scale > 1.0:
        scale *= 2
    return w / scale, scale

def to_q15(x):
    return np.clip(np.round(x * 32768.0), -32768, 32767).astype(np.int64)

def to_q15_bias(x):
    return np.clip(np.round(x * 32768.0), -2**31, 2**31 - 1).astype(np.int64)

def quantize_weights_flatten(model):
    conv_layers = [l for l in model.layers if 'conv1d' in l.name]
    dense_layers = [l for l in model.layers if l.name.startswith('dense')]
    c1_w_f, c1_b_f = conv_layers[0].get_weights()
    c2_w_f, c2_b_f = conv_layers[1].get_weights()
    c3_w_f, c3_b_f = conv_layers[2].get_weights()
    d_w_f, d_b_f = dense_layers[0].get_weights()
    fc_w_f, fc_b_f = dense_layers[1].get_weights()
    c1_w = np.transpose(c1_w_f[:, 0, :], (1, 0))
    c2_w = np.transpose(c2_w_f, (2, 1, 0))
    c3_w = np.transpose(c3_w_f, (2, 1, 0))
    d_w = d_w_f.T
    fc_w = fc_w_f.T
    out = {}
    for name, w, b in [('conv1', c1_w, c1_b_f), ('conv2', c2_w, c2_b_f), ('conv3', c3_w, c3_b_f),
                        ('dense', d_w, d_b_f), ('fc2', fc_w, fc_b_f)]:
        w_s, scale = scale_for_q15(w)
        out[f'{name}_w'] = to_q15(w_s)
        out[f'{name}_b'] = to_q15_bias(b / scale)
    return out

def windows_1d(a, out_len, k, stride, axis):
    a = np.ascontiguousarray(a)
    shape = list(a.shape); shape[axis] = out_len; shape = shape + [k]
    strides = list(a.strides); ts = strides[axis]
    strides[axis] = ts * stride; strides = strides + [ts]
    return as_strided(a, shape=shape, strides=strides)

def q15_forward(qw, X, s1, s2, s3, seq_len):
    c1w, c1b = qw['conv1_w'].astype(np.int64), qw['conv1_b'].astype(np.int64)
    c2w, c2b = qw['conv2_w'].astype(np.int64), qw['conv2_b'].astype(np.int64)
    c3w, c3b = qw['conv3_w'].astype(np.int64), qw['conv3_b'].astype(np.int64)
    dw, db = qw['dense_w'].astype(np.int64), qw['dense_b'].astype(np.int64)
    fcw, fcb = qw['fc2_w'].astype(np.int64), qw['fc2_b'].astype(np.int64)
    F1, K1 = c1w.shape
    F2, _, K2 = c2w.shape
    F3, _, K3 = c3w.shape
    c1out = (seq_len - K1) // s1 + 1
    c2out = (c1out - K2) // s2 + 1
    c3out = (c2out - K3) // s3 + 1
    Xq = np.clip(np.round(X * 32768), -32768, 32767).astype(np.int64)
    win1 = windows_1d(Xq, c1out, K1, s1, axis=1)
    a1 = np.maximum(0, (np.einsum('ntk,fk->nft', win1, c1w) >> 15) + c1b[None, :, None])
    a1c = np.clip(a1, -32768, 32767)
    win2 = windows_1d(a1c, c2out, K2, s2, axis=2)
    a2 = np.maximum(0, (np.einsum('nctk,fck->nft', win2, c2w) >> 15) + c2b[None, :, None])
    a2c = np.clip(a2, -32768, 32767)
    win3 = windows_1d(a2c, c3out, K3, s3, axis=2)
    a3 = np.maximum(0, (np.einsum('nctk,fck->nft', win3, c3w) >> 15) + c3b[None, :, None])
    a3c = np.clip(a3, -32768, 32767)
    N = X.shape[0]
    flat = a3c.reshape(N, F3 * c3out)
    hidden = np.clip(np.maximum(0, (flat @ dw.T >> 15) + db[None, :]), -32768, 32767)
    logits = (hidden @ fcw.T >> 15) + fcb[None, :]
    return np.argmax(logits, axis=1)

## Evaluate (Q15)

In [12]:
qw = quantize_weights_flatten(cp_model)
X_test_flat = X_test.reshape(-1, SEQ_LEN)
y_pred_q15 = q15_forward(qw, X_test_flat, 10, 1, 1, SEQ_LEN)

print(classification_report(y_test, y_pred_q15, target_names=list(CLASSES), digits=4))

              precision    recall  f1-score   support

      benign     0.9406    0.9920    0.9656       623
     malware     0.9909    0.9331    0.9611       583

    accuracy                         0.9635      1206
   macro avg     0.9658    0.9625    0.9634      1206
weighted avg     0.9649    0.9635    0.9635      1206



In [13]:
conf_matrix_q15 = confusion_matrix(y_test, y_pred_q15)
conf_matrix_q15_df = pd.DataFrame(conf_matrix_q15, index=list(CLASSES), columns=list(CLASSES))
print("Confusion Matrix (Q15):")
print(conf_matrix_q15_df)

Confusion Matrix (Q15):
         benign  malware
benign      618        5
malware      39      544
